## Introduction

This script to test the word introduction methods. 

Word introduction method: 
- takes meeting transcript paragraphs that are sequentially ordered
- Creates a set to keep words that it has already seen
- Ingests each paragragraph and compares against the set of already-seen words, if there are words not in the set of already-seen words, it calls those "novel" words
- If it detects novel words, then it looks to see if the speaker is from an LMC or UMC country
- If there is a novel word + LMC or UMC country, then it looks forward a set number of paragraphs (default 500), and looks to see if the word is "adopted"
        - it determines "adopted" by whether the word is used in the forward-looking window (500)
        - If the word appears in the foward-looking window, the algorithm saves: countries who used the word, where it was used, how often it was used

Next steps for generalization:
    - Expand to look at comparative rates of adoption by all income levels?
    - Go from descriptive to something with a statistical significance?


Change point analysis:

Note that as implemented, the change point is descriptive-- looks for a basically arbrary cutoff (aka: bottom quartile of cosine similarities). Scaling it up could look like switching from a threshold to a formal framework, like `ruptures` library in Python, Bayesian change point models, CUSUM tests.
- Ingests transcripts, orders sequentially by meeting and within-meeting position
- Takes the 10 paragraphs before and 10 paragraphs after the input paragraph
- Compares vocabulary in the before and after set via a TF-IDF matrix that is capped at 1k tokens (minus automatically dropped stopwords)
- Compares cosine similarity for the before and after sets
- Saves all cosine similarities for the before/after pairs for each rolling paragraph
- Operationalizes "change points" as the bottom quartile of cosine simlarities
- Looks for whether those are associated with LMC or UMC speakers by taking identified change points, 
and looking back for whether an LMC or UMC speaker was in the most recent 5 paragraphs (note that this means probably want to take out the agenda content, or it will probably return change point at the start of a meeting)

Formalizing the change point detection:
- Move from descriptive & threshold-based to a statistical model, like Bayesian changepoint models or the `ruptures` package in Python
- Use logistic regression to predict UMC/LMC speakers before an identified change point
- Use survival models to model the "hazard" rate of a change point happening conditional on speaker income categorization
- Randomization inference: shuffle speaker identifies for the paragraphs, and recompute whether income level is still statistically associated with changes; observe whether the original association is stronger than chance
- use false discovery rate correction to account for multiple testing
- Robustness tests:
        - Vary window from 10: 5, 15, 20, 50

In [10]:
import pandas as pd
import numpy as np
from collections import defaultdict, Counter
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import matplotlib.pyplot as plt

import os
from pathlib import Path
import glob ## file path processing

from word_introduction_helpers import VocabularyDiffusionDetector # leadership via word introduction
from word_introduction_helpers import DiscussionPatternDetector # vocabulary change point analysis
from word_introduction_helpers import analyze_leadership_through_change_points #analysis wrapper

In [2]:
DATA_DIR = Path('../data/')

In [3]:
wtodatapath = DATA_DIR / 'wtoCTDSpeakerParagraphMto117_varyingInc.csv'

wtodata = pd.read_csv(wtodatapath)

In [4]:
print(wtodata.columns)
print(wtodata.head)

Index(['Unnamed: 0', 'paranum', 'paratext', 'doc', 'firstent', 'ents',
       'speaker.change', 'isagenda', 'minutes.meta', 'behalf', 'date', 'year',
       'meetingno', 'numdate', 'people', 'codes', 'Freq', 'country', 'region',
       'inc_level_abbrev', 'pres.speaker', 'pid', 'dynamic_income'],
      dtype='object')
<bound method NDFrame.head of        Unnamed: 0  paranum                                           paratext  \
0               0      1.0  1.      At the first Session of the Committee ...   
1               1      2.0  2.       The Chairman said that paragraph 2 of...   
2               2      3.0  3.       The Chairman then went on to other pr...   
3               3      4.0  4.       The Chairman went on to the proposed ...   
4               4      5.0  5.       The Chairman said that in accordance ...   
...           ...      ...                                                ...   
11732       11732    107.0  107. The representative of the European Union ...   
11

In [5]:
## Organize the data to sequence by meeting then posistion in meeting:

In [6]:
wtodata = wtodata.sort_values(['year', 'meetingno', 'paranum'])

In [7]:
print(wtodata.columns)
print(wtodata['dynamic_income'].unique())

Index(['Unnamed: 0', 'paranum', 'paratext', 'doc', 'firstent', 'ents',
       'speaker.change', 'isagenda', 'minutes.meta', 'behalf', 'date', 'year',
       'meetingno', 'numdate', 'people', 'codes', 'Freq', 'country', 'region',
       'inc_level_abbrev', 'pres.speaker', 'pid', 'dynamic_income'],
      dtype='object')
['Nonstate' 'Lower middle income' 'High income' 'Upper middle income'
 'Low income' 'Aggregated']


## Testing out basic change point and influence detection 
This is the model that creates a window to see when words are adopted; and runs changepoint analysis via cosine similarity

In [11]:
results = analyze_leadership_through_change_points(wtodata, 
                                                   inc_cat = "Aggregated") ##initial design finds no potential leadership moments
## 

Found 101 instances of identified income categories introducing words that got adopted

Top vocabulary innovations:
    year  novel_word  num_adopters  \
66  1999       frame            16   
92  2019       membe            15   
16  1995        east            12   
98  2022     ukraine            12   
2   1995        some            10   
33  1996         saw            10   
94  2019        prop            10   
19  1995     studies             9   
1   1995       where             7   
40  1996  technology             7   

                                  adopting_categories  
66  [Upper middle income, Lower middle income, Non...  
92  [Lower middle income, High income, Upper middl...  
16  [Lower middle income, High income, Nonstate, L...  
98  [Upper middle income, Lower middle income, Hig...  
2        [Lower middle income, High income, Nonstate]  
33  [Lower middle income, High income, Upper middl...  
94  [Upper middle income, Lower middle income, Hig...  
19  [Lower middle